# Task 2: Time Series Forecasting Models for Tesla (TSLA)

This notebook builds ARIMA and LSTM forecasting models using the cleaned TSLA time series data. It compares model performance and prepares predictions for the test period.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.data_loader import load_or_fetch_assets, save_processed_data, prepare_time_series
from src.eda import compute_daily_returns
from src.forecasting import train_arima, arima_forecast, train_lstm, lstm_predict, evaluate_forecast

start_date = "2015-01-01"
end_date = "2026-06-30"
ticker = "TSLA"

raw = load_or_fetch_assets([ticker], start_date, end_date, per_ticker=True, force_fetch=False)
raw_tsla = raw[raw['Ticker'] == ticker].reset_index(drop=True)
raw_tsla = prepare_time_series(raw_tsla)
raw_tsla = compute_daily_returns(raw_tsla)
save_processed_data(raw_tsla, f"{ticker}.csv")

tsla = raw_tsla[["Date", "Adj Close", "Daily Return"]].copy()

train = tsla[tsla["Date"] < "2025-01-01"]
test = tsla[tsla["Date"] >= "2025-01-01"]

train_series = train["Adj Close"]
test_series = test["Adj Close"]

print("Training records:", len(train))
print("Testing records:", len(test))

## ARIMA Model Development

Train an ARIMA model on the TSLA training series and forecast the test period.

In [ ]:
arima_model = train_arima(train_series, seasonal=False, m=1)
arima_order = arima_model.order
print("ARIMA model order:", arima_order)
arima_preds = arima_forecast(arima_model, len(test_series))

arima_results = evaluate_forecast(test_series.values, arima_preds)
print("ARIMA performance:", arima_results)

## LSTM Model Development

Train an LSTM model using the past 60 days of closing prices to predict the next day.

In [ ]:
window_size = 60
lstm_model, scaler = train_lstm(train_series, window_size=window_size, epochs=20, batch_size=32)

concat_series = pd.concat([train_series, test_series]).reset_index(drop=True)
lstm_preds = lstm_predict(lstm_model, scaler, concat_series, window_size=window_size)
lstm_preds = lstm_preds[-len(test_series):]

lstm_results = evaluate_forecast(test_series.values, lstm_preds)
print("LSTM performance:", lstm_results)

## Forecast Comparison

Compare actual TSLA prices with ARIMA and LSTM predictions over the test period.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test["Date"], test_series.values, label="Actual TSLA")
plt.plot(test["Date"], arima_preds, label="ARIMA Predictions")
plt.plot(test["Date"], lstm_preds, label="LSTM Predictions")
plt.title("TSLA Test Period Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("Adj Close")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Model Performance Summary

Summarize the model metrics and identify the stronger forecasting approach.

In [ ]:
performance_comparison = pd.DataFrame([
    {"Model": "ARIMA", **arima_results},
    {"Model": "LSTM", **lstm_results},
])
performance_comparison